In [1]:
import pandas as pd
import numpy as np
import joblib

# ── 1. CHARGEMENT DES OUTILS SAUVEGARDÉS ──────────────────────────────────────
# Assurez-vous que les chemins vers vos fichiers .pkl sont corrects
modele_charge = joblib.load('../models/meilleur_modeleL.pkl')
preprocessor_charge = joblib.load('../models/preprocessor.pkl')
# features_selectionnees = ['temperature_2m_mean', 'wind_speed_m_s', 'stagnation_index', ...]
features_selectionnees = joblib.load('../models/features_importantes.joblib')

In [2]:
def apply_feature_engineering(df_input):
    """
    Applique l'ensemble du pipeline de Feature Engineering pour le projet PM2.5.
    Garantit la cohérence entre le train set et les données de validation 2026.
    """
    # Copie pour éviter de modifier le dataframe original par référence
    df = df_input.copy()
    
    # S'assurer que 'time' est au format datetime
    df['time'] = pd.to_datetime(df['time'])
    
    # ── 1. Calendrier et Temporel ─────────────────────────────────────────────
    df['mois'] = df['time'].dt.month
    df['annee'] = df['time'].dt.year
    df['mois_nom'] = df['time'].dt.month_name().str[:4]
    df['forte_pluie'] = df['precipitation_sum'] > 20
    
    # Calculs thermiques et hydriques
    df['amplitude_thermique'] = df['temperature_2m_max'] - df['temperature_2m_min']
    df['ecart_ressenti'] = df['apparent_temperature_mean'] - df['temperature_2m_mean']
    df['bilan_hydrique'] = df['precipitation_sum'] - df['et0_fao_evapotranspiration']

    # Saisonnalité et Cyclicités
    df['quarter'] = df['time'].dt.quarter
    df['day_of_year'] = df['time'].dt.dayofyear
    
    df['month_sin'] = np.sin(2 * np.pi * df['mois'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['mois'] / 12)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

    # Saison sèche (Nov à Mar) et Week-end
    df['is_dry_season'] = df['mois'].isin([11, 12, 1, 2, 3]).astype(int)
    df['is_weekend'] = df['time'].dt.dayofweek.isin([5, 6]).astype(int)

    # ── 2. Variables dérivées (Physique de l'air) ──────────────────────────────
    #df['sunshine_ratio'] = df['sunshine_duration'] / (df['daylight_duration'] + 1e-6)
    df['is_no_wind'] = (df['wind_speed_10m_max'] < 5).astype(int)
    df['is_no_rain'] = (df['precipitation_sum'] < 0.1).astype(int)
    df['gust_ratio'] = df['wind_gusts_10m_max'] / (df['wind_speed_10m_max'] + 1e-6)
    df['heat_stress'] = df['temperature_2m_mean'] * df['et0_fao_evapotranspiration']
    df['is_hot_day'] = (df['temperature_2m_max'] >= 35).astype(int)
    df['is_heavy_rain'] = (df['precipitation_sum'] >= 20).astype(int)
    df['temp_per_radiation'] = df['temperature_2m_mean'] / (df['shortwave_radiation_sum'] + 1e-6)
    df['stagnation_index'] = ((df['wind_speed_10m_max'] < 15) & (df['precipitation_sum'] == 0)).astype(int)

    # ── 3. Séries temporelles (Lags et Rolling) ───────────────────────────────
    # Tri par ville et par temps indispensable pour les calculs glissants
    df = df.sort_values(['city', 'time']).reset_index(drop=True)

    lags = [1, 3, 7]
    for lag in lags:
        df[f'temp_lag{lag}'] = df.groupby('city')['temperature_2m_mean'].shift(lag)
        df[f'wind_lag{lag}'] = df.groupby('city')['wind_speed_10m_max'].shift(lag)
        df[f'precip_lag{lag}'] = df.groupby('city')['precipitation_sum'].shift(lag)
        df[f'wind_dir_lag{lag}'] = df.groupby('city')['wind_direction_10m_dominant'].shift(lag)
        df[f'sunshine_lag{lag}'] = df.groupby('city')['sunshine_duration'].shift(lag)

    # Moyennes et cumuls mobiles
    df['temp_roll7'] = df.groupby('city')['temperature_2m_mean'].transform(lambda x: x.rolling(7, min_periods=1).mean())
    df['precip_roll7'] = df.groupby('city')['precipitation_sum'].transform(lambda x: x.rolling(7, min_periods=1).mean())
    df['wind_roll7'] = df.groupby('city')['wind_speed_10m_max'].transform(lambda x: x.rolling(7, min_periods=1).mean())
    df['temp_roll30'] = df.groupby('city')['temperature_2m_mean'].transform(lambda x: x.rolling(30, min_periods=1).mean())
    
    df['temp_anomaly'] = df['temperature_2m_mean'] - df['temp_roll30']
    df['precip_cumul7'] = df.groupby('city')['precipitation_sum'].transform(lambda x: x.rolling(7, min_periods=1).sum())
    df['precip_cumul3'] = df.groupby('city')['precipitation_sum'].transform(lambda x: x.rolling(3, min_periods=1).sum())

    # ── 4. Catégorisation Météo ───────────────────────────────────────────────
    def categorize_weather(code):
        if code == 0: return 'Ciel dégagé'
        elif code in [1, 2]: return 'Nuageux'
        elif code == 3: return 'Couvert'
        elif code in [51, 53, 55, 61, 63, 65]: return 'Pluie/Bruine'
        else: return 'Inconnu'

    df['weather_categorie'] = df['weather_code'].apply(categorize_weather)
    
    ordre_impact = ['Ciel dégagé', 'Nuageux', 'Couvert', 'Pluie/Bruine']
    impact_map = {cat: i for i, cat in enumerate(ordre_impact)}
    df['weather_encoded'] = df['weather_categorie'].map(impact_map).fillna(-1)

    return df

# Exemple d'utilisation :
# df_final = apply_feature_engineering(df_raw)

In [6]:
# ── 2. CHARGEMENT ET PRÉPARATION DES NOUVELLES DONNÉES (2026) ─────────────────

# données météo brutes de 2026
df_new = pd.read_csv('climat_air_quality_model.csv')

# Application de la fonction de Feature Engineering

df_processed = apply_feature_engineering(df_new)

# ── 3. PRÉTRAITEMENT (SCALING & ENCODAGE) ─────────────────────────────────────

# On définit X_new en retirant les colonnes qui n'étaient pas des features d'entrée

X_new = df_processed.drop(['id', 'time', 'pm25_proxy'], axis=1, errors='ignore')

# Transformation avec le preprocessor chargé (StandardScaler + OneHotEncoder)
X_new_prepared = preprocessor_charge.transform(X_new)

# ── 4. RECONSTITUTION ET SÉLECTION DES FEATURES IMPORTANTES ───────────────────

# Récupération des noms de toutes les colonnes après transformation
cat_encoder = preprocessor_charge.named_transformers_['cat'].named_steps['encoder']
noms_num = X_new.select_dtypes(include=['int64', 'float64']).columns.tolist()
noms_cat = cat_encoder.get_feature_names_out().tolist()
tous_les_noms_features = noms_num + noms_cat

# Création d'un DataFrame temporaire pour filtrer facilement
df_reconstitue = pd.DataFrame(X_new_prepared, columns=tous_les_noms_features)

# Garder uniquement les colonnes que le modèle connaît
X_final_2026 = df_reconstitue[features_selectionnees]

# ── 5. PRÉDICTIONS ────────────────────────────────────────────────────────────

predictions_2026 = modele_charge.predict(X_final_2026)

# Ajout des prédictions au DataFrame original pour l'affichage
df_processed['predictions_pm25'] = predictions_2026

# ── 6. AFFICHAGE DES RÉSULTATS ────────────────────────────────────────────────

print("--- Prédictions terminées ---")
print(df_processed[['time', 'city', 'pm25_proxy', 'predictions_pm25']].head())

# Comparaison de la performance réelle / Validation
from sklearn.metrics import r2_score, mean_absolute_error

if 'pm25_proxy' in df_processed.columns:
    mask = ~df_processed['pm25_proxy'].isna() # On ignore les lignes sans cible
    r2 = r2_score(df_processed.loc[mask, 'pm25_proxy'], df_processed.loc[mask, 'predictions_pm25'])
    mae = mean_absolute_error(df_processed.loc[mask, 'pm25_proxy'], df_processed.loc[mask, 'predictions_pm25'])
    print(f"\nPerformance sur données 2026 :")
    print(f"R² Score : {r2:.4f}")
    print(f"MAE      : {mae:.4f}")

KeyError: 'weather_code'